# AutoGen Group Chat Demo

Conversational multi-agent workflows with AutoGen: role conditioning, group chat orchestration, and termination.

| Primitive | What it is | Why it is here |
|-----------|------------|----------------|
| `AssistantAgent` | An LLM-backed role with a `system_message` pinning its job | Role conditioning: each agent stays in one lane |
| `UserProxyAgent` | A stand-in user / executor that can run code or relay input | Closes the loop when an agent needs an action taken |
| `RoundRobinGroupChat` | Group chat that rotates speakers in a fixed order | Deterministic turn-taking, ideal for planner / coder / reviewer chains |
| `SelectorGroupChat` | Group chat where a model picks the next speaker each turn | Dynamic routing when the right agent depends on the current message |
| `TextMentionTermination` | Termination condition fired when a token (e.g. `'TERMINATE'`) appears | Lets agents themselves decide when the chat is done |
| `MaxMessageTermination` | Hard cap on number of messages | Safety net so a stuck conversation cannot loop forever |
| `OpenAIChatCompletionClient` | AutoGen's model client for OpenAI-compatible APIs | One-class swap to Gemini (OpenAI-compatible endpoint) or Anthropic |

We default to **OpenAI's `gpt-4o-mini`** because the AutoGen group chat orchestration leans on stable structured responses for termination and selector decisions, and the Gemini and Anthropic lines sit directly under the active one as commented swap-points. 

Likewise we keep every termination condition explicit and combine `TextMentionTermination` with `MaxMessageTermination` so neither a chatty agent nor a silent one can hang the demo.

## Setup

In [1]:
!pip install -q "autogen-agentchat" "autogen-ext[openai]" "autogen-ext[anthropic]" python-dotenv

from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.teams import RoundRobinGroupChat, SelectorGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
import os, asyncio

load_dotenv()


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


True

Let's first define the model client. The default below runs on **OpenAI** (`gpt-4o-mini`); the commented blocks underneath are exact swap-points for Gemini (via the OpenAI-compatible endpoint) and Anthropic.

In [2]:
# Default wiring (swap the block below to change backend)
model_client = OpenAIChatCompletionClient(model='gpt-4o-mini', temperature=0)

# Alternatives

# # Gemini via OpenAI-compatible endpoint
# model_client = OpenAIChatCompletionClient(
#     model='gemini-2.5-flash',
#     base_url='https://generativelanguage.googleapis.com/v1beta/openai/',
#     api_key=os.environ['GEMINI_API_KEY'],
# )
#

# # Anthropic
# from autogen_ext.models.anthropic import AnthropicChatCompletionClient
# model_client = AnthropicChatCompletionClient(model='claude-haiku-4-5', temperature=0)

print(f'Model client ready: {model_client.__class__.__name__}')

Model client ready: OpenAIChatCompletionClient


*Tip: AutoGen's agent and team APIs are [`async`](https://docs.python.org/3/library/asyncio.html). Inside Jupyter we use `await` directly because the notebook event loop is already running; outside Jupyter we wrap calls in `asyncio.run(...)`. Both patterns are shown below.*

## Role conditioning with `AssistantAgent`

Now let's build the smallest possible AutoGen unit — a single `AssistantAgent` pinned to one role via its `system_message`.

Now, we will
- declare a `PLANNER_SYSTEM` prompt that pins the agent to *planner only — no code, no review*,
- instantiate one `AssistantAgent` with that `system_message` and the shared `model_client`,
- send it a single message via `on_messages` and read the reply,
- confirm the agent stays in role even when nudged toward a different job.

In [3]:
PLANNER_SYSTEM = """You are the PLANNER on a small engineering team.

Your one job: read a coding task and produce a numbered, 3-step plan.
Constraints:
- Do NOT write code. Do NOT review code. Plan only.
- Each step is one short imperative sentence.
- If the request is outside planning, reply with a polite refusal.
"""

planner_solo = AssistantAgent(
    name='planner',
    model_client=model_client,
    system_message=PLANNER_SYSTEM,
)
print(f'planner role pinned via system_message ({len(PLANNER_SYSTEM)} chars).')

planner role pinned via system_message (298 chars).


The `system_message` **is** the role contract. AutoGen replays it on every turn, so the agent cannot drift mid-conversation. Let's exercise it with one in-role request and confirm the reply respects the *plan only — no code* constraint.

In [4]:
from autogen_core import CancellationToken

async def ask_solo(agent: AssistantAgent, prompt: str) -> str:
    """Send one TextMessage to a single agent and return its reply content."""
    response = await agent.on_messages(
        [TextMessage(content=prompt, source='user')],
        cancellation_token=CancellationToken(),
    )
    return response.chat_message.content

reply_in_role = await ask_solo(planner_solo,
    'Write a Python function that returns the n-th Fibonacci number.')
print('---- planner (in role) ----')
print(reply_in_role[:120] + '...')

---- planner (in role) ----
1. Research the Fibonacci sequence and determine the most efficient method to calculate the n-th number (e.g., iterative...


## Three roles: planner, coder, reviewer

Now let's add two more `AssistantAgent`s, each pinned to a complementary role. These three agents are the cast for the rest of the notebook.

Now, we will
- declare `CODER_SYSTEM` and `REVIEWER_SYSTEM` prompts, each pinning a distinct job,
- instantiate `coder` and `reviewer` `AssistantAgent`s sharing the same `model_client`,
- give the reviewer the explicit instruction to emit `'TERMINATE'` when the code looks correct,
- print a one-line summary listing the cast and their roles.

In [5]:
CODER_SYSTEM = """You are the CODER on a small engineering team.

Your one job: take the planner's plan and write Python code that implements it.
Constraints:
- Output runnable Python in a single fenced ```python``` block.
- Do NOT plan. Do NOT review your own code. Coding only.
- If the plan is unclear, ask the planner exactly one clarifying question.
"""

REVIEWER_SYSTEM = """You are the REVIEWER on a small engineering team.

Your one job: read the coder's code and either approve it or request a fix.
Constraints:
- If the code is correct and complete, reply with a one-sentence approval that ENDS with the literal word TERMINATE.
- If the code has a bug, describe the bug in one sentence and tell the coder what to change.
- Do NOT write the fix yourself. Do NOT plan. Review only.
"""

Now let's instantiate the three role-conditioned agents that the group chats below will reuse. Same `model_client`, three distinct `system_message`s — that is the whole role conditioning surface.

In [6]:
planner = AssistantAgent(
    name='planner',
    model_client=model_client,
    system_message=PLANNER_SYSTEM,
)
coder = AssistantAgent(
    name='coder',
    model_client=model_client,
    system_message=CODER_SYSTEM,
)
reviewer = AssistantAgent(
    name='reviewer',
    model_client=model_client,
    system_message=REVIEWER_SYSTEM,
)

for a in [planner, coder, reviewer]:
    print(f'  agent={a.name:<8}  role conditioning via system_message len={len(a._system_messages[0].content)}')

  agent=planner   role conditioning via system_message len=298
  agent=coder     role conditioning via system_message len=337
  agent=reviewer  role conditioning via system_message len=409


## Termination conditions

Now let's pick the rule that decides WHEN the group chat is done. Termination is a first-class object in AutoGen, and conditions compose.

Now, we will
- build a `TextMentionTermination('TERMINATE')` that fires the moment any agent emits the literal token,
- build a `MaxMessageTermination(8)` that caps the chat at eight messages no matter what,
- combine them with `|` so EITHER condition ends the chat — model-driven exit OR safety net,
- print the composite condition class so we can see the boolean wiring AutoGen actually uses.

In [7]:
text_term = TextMentionTermination('TERMINATE')
max_term  = MaxMessageTermination(8)

# Compose: chat ends as soon as EITHER condition fires.
termination = text_term | max_term

print('---- termination wiring ----')
print(f'  text mention : {text_term.__class__.__name__}("TERMINATE")')
print(f'  max messages : {max_term.__class__.__name__}(8)')
print(f'  composite    : {termination.__class__.__name__}')

---- termination wiring ----
  text mention : TextMentionTermination("TERMINATE")
  max messages : MaxMessageTermination(8)
  composite    : OrTerminationCondition


*Tip: always pair a model-driven termination (the agent says `TERMINATE`) with a hard cap (`MaxMessageTermination`). If the reviewer's role conditioning fails to emit the token, the cap still terminates the chat — without it, an agent loop can spin until your model budget runs out.*

## `RoundRobinGroupChat` capstone

Now let's wire the three agents into a `RoundRobinGroupChat` and watch them solve a coding task in fixed planner -> coder -> reviewer order.

Now, we will
- construct a `RoundRobinGroupChat([planner, coder, reviewer], termination_condition=termination)`,
- run it on a small coding task with `await chat.run(task=...)`,
- print every turn under a `---- TURN N (<agent>) ----` banner with the message content truncated,
- print the final reviewer verdict under a `=== FINAL ===` banner.

In [8]:
TASK = (
    'Write a Python function `is_prime(n: int) -> bool` that returns whether n '
    'is a prime number. Cover the edge cases n < 2.'
)

round_robin_chat = RoundRobinGroupChat(
    [planner, coder, reviewer],
    termination_condition=termination,
)
print('group chat assembled with', len(round_robin_chat._participants), 'agents')

group chat assembled with 3 agents


Now the run. We invoke the chat with `await` (Jupyter handles top-level await), then walk the returned `messages` list and print each turn under a four-dash banner. The reviewer's `TERMINATE` token should fire well before the eight-message cap.

In [9]:
result = await round_robin_chat.run(task=TASK)

print('=== TASK ===')
print(TASK)
print()
for i, msg in enumerate(result.messages, start=1):
    src = getattr(msg, 'source', 'system')
    body = getattr(msg, 'content', '')
    if not isinstance(body, str):
        body = str(body)
    print(f'---- TURN {i} ({src}) ----')
    print(body[:120] + '...' if len(body) > 120 else body)
    print()

final = result.messages[-1]
final_body = getattr(final, 'content', '') or ''
if not isinstance(final_body, str):
    final_body = str(final_body)
print('=== FINAL ===')
print(f'stop_reason: {result.stop_reason}')
print(f'turns      : {len(result.messages)}')
print(f'last       : [{getattr(final, "source", "?")}] {final_body[:120] + "..." if len(final_body) > 120 else final_body}')

=== TASK ===
Write a Python function `is_prime(n: int) -> bool` that returns whether n is a prime number. Cover the edge cases n < 2.

---- TURN 1 (user) ----
Write a Python function `is_prime(n: int) -> bool` that returns whether n is a prime number. Cover the edge cases n < 2.

---- TURN 2 (planner) ----
1. Define the function `is_prime` that takes an integer `n` as an argument.  
2. Implement a check for edge cases where ...

---- TURN 3 (coder) ----
```python
import math

def is_prime(n: int) -> bool:
    if n < 2:
        return False
    for i in range(2, int(math.s...

---- TURN 4 (reviewer) ----
The code correctly implements the prime number check and handles edge cases appropriately, so I approve it. TERMINATE

=== FINAL ===
stop_reason: Text 'TERMINATE' mentioned
turns      : 4
last       : [reviewer] The code correctly implements the prime number check and handles edge cases appropriately, so I approve it. TERMINATE


## `SelectorGroupChat`

Now let's swap the deterministic round-robin for a `SelectorGroupChat`, where a model decides who speaks next based on the current message.

Now, we will
- declare a `SELECTOR_PROMPT` that tells the selector model how to choose between planner, coder, and reviewer,
- construct a `SelectorGroupChat` reusing the same three agents and the same composite termination,
- run it on a slightly different task so the selector has to actually route, not just rotate,
- print every turn under a `---- TURN N (<agent>) ----` so we can see the dynamic order.

In [10]:
SELECTOR_PROMPT = """You orchestrate a small engineering team: planner, coder, reviewer.

Read the conversation so far and pick the SINGLE agent who should speak next.

Routing rules:
- If there is no plan yet, pick the planner.
- If there is a plan but no code yet, pick the coder.
- If there is code that has not been reviewed, pick the reviewer.
- If the reviewer asked for a fix, pick the coder.

Reply with exactly one of: planner, coder, reviewer.
"""

selector_chat = SelectorGroupChat(
    [planner, coder, reviewer],
    model_client=model_client,
    selector_prompt=SELECTOR_PROMPT,
    termination_condition=termination,
    allow_repeated_speaker=True,
)
print('selector group chat assembled; routing decided per-turn by the model')

selector group chat assembled; routing decided per-turn by the model


Now the selector run. Notice how the speaker order is no longer fixed — when the reviewer requests a fix, the selector should route back to the coder rather than blindly advancing to the planner. That is the entire selector value over round-robin.

In [11]:
SELECTOR_TASK = (
    'Write a Python function `factorial(n: int) -> int` that returns n! for n >= 0 '
    'and raises ValueError for negative n.'
)

selector_result = await selector_chat.run(task=SELECTOR_TASK)

print('=== TASK ===')
print(SELECTOR_TASK)
print()
for i, msg in enumerate(selector_result.messages, start=1):
    src = getattr(msg, 'source', 'system')
    body = getattr(msg, 'content', '')
    if not isinstance(body, str):
        body = str(body)
    print(f'---- TURN {i} ({src}) ----')
    print(body[:120] + '...' if len(body) > 120 else body)
    print()

print('=== FINAL ===')
print(f'stop_reason: {selector_result.stop_reason}')
print(f'turns      : {len(selector_result.messages)}')
speakers = [getattr(m, 'source', '?') for m in selector_result.messages]
print(f'speaker order: {speakers}')

=== TASK ===
Write a Python function `factorial(n: int) -> int` that returns n! for n >= 0 and raises ValueError for negative n.

---- TURN 1 (user) ----
Write a Python function `factorial(n: int) -> int` that returns n! for n >= 0 and raises ValueError for negative n.

---- TURN 2 (planner) ----
1. Define the function `factorial` that takes an integer `n` as an argument.  
2. Implement a check to raise a `ValueErr...

---- TURN 3 (planner) ----
1. Define the function `factorial` that takes an integer `n` as an argument.  
2. Implement a check to raise a `ValueErr...

---- TURN 4 (planner) ----
1. Define the function `factorial` that takes an integer `n` as an argument.  
2. Implement a check to raise a `ValueErr...

---- TURN 5 (planner) ----
1. Define the function `factorial` that takes an integer `n` as an argument.  
2. Implement a check to raise a `ValueErr...

---- TURN 6 (planner) ----
1. Define the function `factorial` that takes an integer `n` as an argument.  
2. Implement a 

## Re-conditioning: change one role and re-run

Now let's prove role conditioning is doing the work by changing the reviewer's `system_message` and re-running the same chat. Nothing else moves.

Now, we will
- build a STRICT reviewer with a `system_message` demanding type hints, a docstring, and at least three test cases,
- assemble a fresh `RoundRobinGroupChat` with the strict reviewer in place of the original,
- run it on the SAME task as the round-robin capstone above,
- print the final coder output and confirm it now carries type hints and tests — a behaviour change driven only by `system_message`.

In [12]:
STRICT_REVIEWER_SYSTEM = """You are the REVIEWER on a small engineering team. You are STRICT.

Your one job: read the coder's code and either approve it or request a fix.
Constraints:
- The code MUST include type hints on every function signature.
- The code MUST include a one-line docstring on every function.
- The code MUST include at least 3 test cases as `assert` statements.
- Only when ALL of the above are present, reply with a one-sentence approval that ENDS with the literal word TERMINATE.
- Otherwise list the missing items in one sentence and tell the coder what to change.
- Do NOT write the fix yourself. Review only.
"""

strict_reviewer = AssistantAgent(
    name='reviewer',
    model_client=model_client,
    system_message=STRICT_REVIEWER_SYSTEM,
)

strict_termination = TextMentionTermination('TERMINATE') | MaxMessageTermination(10)

strict_chat = RoundRobinGroupChat(
    [planner, coder, strict_reviewer],
    termination_condition=strict_termination,
)
print('reassembled chat with STRICT reviewer; same planner, same coder, same task')

reassembled chat with STRICT reviewer; same planner, same coder, same task


Now the re-run. We are expecting a longer conversation — the strict reviewer should reject the coder's first attempt for missing tests or hints, the coder should patch, and only THEN should the chat terminate. The behaviour change comes from one `system_message` swap.

In [13]:
strict_result = await strict_chat.run(task=TASK)

print('=== TASK (re-run with STRICT reviewer) ===')
print(TASK)
print()
for i, msg in enumerate(strict_result.messages, start=1):
    src = getattr(msg, 'source', 'system')
    body = getattr(msg, 'content', '')
    if not isinstance(body, str):
        body = str(body)
    print(f'---- TURN {i} ({src}) ----')
    print(body[:120] + '...' if len(body) > 120 else body)
    print()

print('=== FINAL ===')
print(f'stop_reason : {strict_result.stop_reason}')
print(f'turns       : {len(strict_result.messages)}')
print(f'role conditioning effect : the only thing that changed was the reviewer system_message')

=== TASK (re-run with STRICT reviewer) ===
Write a Python function `is_prime(n: int) -> bool` that returns whether n is a prime number. Cover the edge cases n < 2.

---- TURN 1 (user) ----
Write a Python function `is_prime(n: int) -> bool` that returns whether n is a prime number. Cover the edge cases n < 2.

---- TURN 2 (planner) ----
1. Define the function `is_prime` that takes an integer `n` as an argument.  
2. Implement a check for edge cases where ...

---- TURN 3 (coder) ----
```python
import math

def is_prime(n: int) -> bool:
    if n < 2:
        return False
    for i in range(2, int(math.s...

---- TURN 4 (reviewer) ----
The code is missing a one-line docstring for the function and at least 3 test cases as `assert` statements. Please add t...

---- TURN 5 (planner) ----
I'm sorry, but I can only assist with planning and cannot modify or review code. If you have a coding task or need a pla...

---- TURN 6 (coder) ----
```python
import math

def is_prime(n: int) -> bool:
    

## `asyncio.run` outside Jupyter

Now let's show the script-friendly form of the same call. Inside Jupyter we used `await` because the kernel already runs an event loop; outside Jupyter we wrap with `asyncio.run(...)`.

Now, we will
- define a small `async def main()` coroutine that runs the round-robin chat on a tiny throwaway task,
- guard the `asyncio.run(...)` call so it only fires in non-Jupyter contexts,
- print the stop reason and the speaker order so the script-style entrypoint is concrete,
- close the model client at the end to release the underlying HTTP connection.

In [14]:
async def main() -> None:
    """Script-style entrypoint mirroring what a `python autogen_demo.py` would do."""
    tiny_chat = RoundRobinGroupChat(
        [planner, coder, reviewer],
        termination_condition=TextMentionTermination('TERMINATE') | MaxMessageTermination(6),
    )
    out = await tiny_chat.run(task='Write a one-line Python function that returns the square of n.')
    print(f'stop_reason : {out.stop_reason}')
    print(f'speakers    : {[getattr(m, "source", "?") for m in out.messages]}')

# In a .py script you would call: asyncio.run(main())
# In Jupyter the kernel already runs an event loop, so we await directly.
await main()

stop_reason : Text 'TERMINATE' mentioned
speakers    : ['user', 'planner', 'coder', 'reviewer']


Do not call `await` inside `asyncio.run(main())` from a Jupyter cell. The kernel already runs an event loop, so `asyncio.run` raises `RuntimeError`. Use `await main()` in notebooks; use `asyncio.run(main())` only in Python scripts.

## Tips and common pitfalls

### Designing AutoGen group chats in production
- Always combine `TextMentionTermination` with `MaxMessageTermination` using `|`. The model-driven exit is your happy path; the message cap is the safety net that bounds your worst-case cost.
- Pin every `AssistantAgent` to one job via `system_message`. Conflated roles (planner who also reviews) are the single biggest source of group chat drift.r both the agents AND the `SelectorGroupChat` selector. Higher temperature increases the rate of off-role replies and erratic routing.
- Reuse one `model_client` across agents instead of constructing one per agent. The client owns the HTTP connection pool; constructing N of them wastes sockets.

### Common pitfalls in role conditioning and termination
- A reviewer that emits `terminate` (lowercase) does NOT trip `TextMentionTermination('TERMINATE')`. Match the casing in the system prompt to the casing in the condition. <br>Mitigation: subclass the condition with a case-insensitive match, or instruct the agent to emit the exact ALL-CAPS token.
- `RoundRobinGroupChat` advances even when the reviewer asks for a fix. If you need *coder -> reviewer -> coder* on rejection, switch to `SelectorGroupChat` with a routing prompt — round-robin is for fixed pipelines, not feedback loops.